# [1/4] NHÁNH CNN – YOLO26s (detection) + Faster R-CNN + VGG16/ResNet50 (classification)
### Đồ án Deep Learning · Active AI Camera – PPE Detection 10 lớp · Chạy file này ĐẦU TIÊN

| Model | Bài toán | HPO |
|---|---|---|
| **YOLO26s** (NMS-free, pretrained COCO) | Detection 10 lớp | lr0 × optimizer (4 cấu hình) → best × 40 epochs |
| **Faster R-CNN ResNet50-FPN** (torchvision) | Detection 10 lớp | baseline 1 cấu hình chuẩn (two-stage tham chiếu) |
| **VGG16 & ResNet50** (Keras, ImageNet) | Classification crop 10 lớp | backbone × lr × dropout (8 cấu hình) → best mỗi backbone |

Kaggle Settings: **GPU T4 x2** + **Internet ON** (bắt buộc — pip cài ultralytics và tải trọng số pretrained COCO/ImageNet) + **+ Add Input dataset `ppe-unified-v1`**. Ước tính ~8–9h trên T4 với ~18k ảnh (QUICK_TEST ~20 phút). Kết quả chuẩn hóa lưu vào `results/summary.csv` cho notebook 04.

## Dataset: PPE_Unified_v1 (dùng chung cho cả 3 notebook — GIỮ NGUYÊN CELL NÀY Ở CẢ 3 FILE)
**Safety-PPE Unified v1** — dataset tự xây dựng bằng cách gộp 3 dataset public (CC BY 4.0): Hard Hat v2 (computer-vision-filqz) + PPE v1 (testcasque) + PPE v5 (mohamed-traore-2ekkp). **12 lớp = 6 cặp tuân thủ/vi phạm**, ~18k ảnh đã: khử trùng lặp theo ảnh gốc, khử trùng chéo MD5, **chuẩn hóa 640×640 letterbox**, chia sẵn **train 70% / val 20% / test 10% theo nhóm ảnh gốc (chống leakage)**. Chi tiết trong `DATASET_CARD.md` của dataset.

**Cách nạp:** dataset đã upload lên Kaggle (`ppe-unified-v1`); ở MỖI notebook bấm **+ Add Input → Your Datasets → ppe-unified-v1**. Không cần API key — dữ liệu nạp offline từ Input. (Lưu ý: Internet ON vẫn cần ở notebook 01/03 để cài thư viện + tải trọng số pretrained.)

Split đã cố định sẵn trong dataset ⇒ cả 3 notebook so sánh trên đúng cùng một phép chia. Xử lý mất cân bằng khi train: (1) repeat-factor oversampling lớp hiếm (train only); (2) undersampling crop ≤ MAX_CROPS_PER_CLASS; (3) class weights trong loss; (4) chọn model theo macro-F1 / mAP từng lớp (lớp có <10 mẫu test — vd no_boots — bị loại khỏi worst_class_f1, báo cáo riêng). Quy tắc: mọi hyperparameter chọn trên VAL; TEST chỉ dùng 1 lần cuối.

In [ ]:
# ====================== CONFIG (CNN) ======================
BRANCH     = "CNN"
QUICK_TEST = False    # True: chạy nhanh kiểm tra pipeline (~15-20 phút)
RUN_HPO    = True

SEED = 42
SPLIT = (0.7, 0.2, 0.1)
RFS_T, REPEAT_CAP = 0.15, 4                 # repeat-factor oversampling lớp hiếm (train only)
CROP_SIZE, CROP_SIZE_VIT = 128, 224
MAX_CROPS_PER_CLASS = 200 if QUICK_TEST else 2500

import os, random, time, shutil, json, numpy as np
random.seed(SEED); np.random.seed(SEED)
WORK = "/kaggle/working"
os.makedirs(f"{WORK}/results", exist_ok=True)
HPO_TABLES = {}

# ============ RESUME: đặt đường dẫn output của lần chạy trước của CHÍNH notebook này (nếu có) ============
RESUME_FROM = ""
if RESUME_FROM and os.path.exists(RESUME_FROM):
    for sub in ["state.json", "results", "runs"]:
        src = os.path.join(RESUME_FROM, sub)
        if os.path.isdir(src): shutil.copytree(src, os.path.join(WORK, sub), dirs_exist_ok=True)
        elif os.path.isfile(src): shutil.copy(src, os.path.join(WORK, sub))
    print("ĐÃ NẠP CHECKPOINT từ:", RESUME_FROM)

STATE_PATH = f"{WORK}/state.json"
STATE = json.load(open(STATE_PATH)) if os.path.exists(STATE_PATH) else {}
def save_state(): json.dump(STATE, open(STATE_PATH, "w"), indent=1, default=str)
def done(key):    return key in STATE
def mark(key, value=True): STATE[key] = value; save_state()
def free():
    import gc, torch; gc.collect(); torch.cuda.empty_cache()

print(f"[{BRANCH}] QUICK_TEST={QUICK_TEST} | RUN_HPO={RUN_HPO} | stage đã xong:", [k for k in STATE] or "(chưa có)")

In [ ]:
%pip install -q -U ultralytics torchmetrics pycocotools
import torch
assert torch.cuda.is_available(), "Chưa bật GPU! Settings -> Accelerator -> GPU T4 x2"
cap = torch.cuda.get_device_capability(0); gpu_name = torch.cuda.get_device_name(0)
if cap[0] < 7:
    raise SystemError(f"GPU {gpu_name} (sm_{cap[0]}{cap[1]}) không được PyTorch hỗ trợ. Đổi sang GPU T4 x2!")
print("GPU:", gpu_name, "- OK")

In [ ]:
from pathlib import Path
from collections import Counter
import yaml, glob

# ==== Tìm PPE_Unified_v1 trong /kaggle/input (đã Add Input) ====
DS_ROOT = None
for c in glob.glob("/kaggle/input/**/data.yaml", recursive=True):
    root = os.path.dirname(c)
    if os.path.isdir(f"{root}/train/images") and os.path.isdir(f"{root}/val/images") and os.path.isdir(f"{root}/test/images"):
        DS_ROOT = root; break
assert DS_ROOT, ("Không tìm thấy PPE_Unified_v1! Upload thư mục PPE_Unified_v1 lên Kaggle "
                 "(Create -> New Dataset) rồi '+ Add Input' vào notebook này.")
print("Dataset input:", DS_ROOT)

raw_yaml = yaml.safe_load(open(f"{DS_ROOT}/data.yaml"))
raw_names = raw_yaml["names"]
if isinstance(raw_names, dict): raw_names = [raw_names[k] for k in sorted(raw_names)]
CLASSES = list(raw_names); NUM_CLS = len(CLASSES)
print(f"{NUM_CLS} lớp:", CLASSES)

# ==== Copy về /kaggle/working (input read-only; YOLO cần ghi labels.cache) ====
# Split ĐÃ cố định sẵn trong dataset (chia theo nhóm ảnh gốc, chống leakage) -> KHÔNG chia lại.
DATA = f"{WORK}/dataset"
CAPS = {"train": 300, "val": 80, "test": 40} if QUICK_TEST else {}
splits = {}
for sp in ["train", "val", "test"]:
    os.makedirs(f"{DATA}/{sp}/images", exist_ok=True); os.makedirs(f"{DATA}/{sp}/labels", exist_ok=True)
    files = sorted(os.listdir(f"{DS_ROOT}/{sp}/images"))
    random.Random(SEED).shuffle(files)
    if sp in CAPS: files = files[:CAPS[sp]]
    done_copy = len(os.listdir(f"{DATA}/{sp}/images")) >= len(files)   # resume: đã copy rồi thì bỏ qua
    for fn in files:
        if done_copy: break
        shutil.copy(f"{DS_ROOT}/{sp}/images/{fn}", f"{DATA}/{sp}/images/{fn}")
        lp = f"{DS_ROOT}/{sp}/labels/{Path(fn).stem}.txt"
        if os.path.exists(lp): shutil.copy(lp, f"{DATA}/{sp}/labels/{Path(fn).stem}.txt")
    splits[sp] = files
print("Số ảnh:", {sp: len(v) for sp, v in splits.items()})

# ==== Kiểm tra đồng nhất kích thước ảnh (dataset đã letterbox 640x640 khi build) ====
from PIL import Image
sample = [f"{DATA}/train/images/{f}" for f in splits["train"][:200]]
dims = Counter(Image.open(p).size for p in sample)
print("Kích thước ảnh (mẫu 200):", dict(dims.most_common(3)))
assert dims.most_common(1)[0][0] == (640, 640), "Ảnh chưa đồng nhất 640x640 - hãy build lại dataset với NORMALIZE_SIZE=640!"

# ==== Thống kê lớp + Repeat-factor oversampling (LVIS) - CHỈ tập train ====
cls_count, img_classes = Counter(), {}
for sp in ["train", "val", "test"]:
    for lf in os.listdir(f"{DATA}/{sp}/labels"):
        if "_rfs" in lf: continue
        present = Counter()
        for line in open(f"{DATA}/{sp}/labels/{lf}"):
            if not line.strip(): continue
            c = int(line.split()[0]); present[c] += 1; cls_count[(sp, CLASSES[c])] += 1
        if sp == "train": img_classes[lf] = present

from math import sqrt
n_train = len(splits["train"])
f_c = {c: sum(1 for s in img_classes.values() if c in s)/max(1, n_train) for c in range(NUM_CLS)}
r_c = {c: max(1.0, sqrt(RFS_T/max(f_c[c], 1e-6))) for c in range(NUM_CLS)}
print("Hệ số lặp lớp hiếm:", {CLASSES[c]: round(r_c[c],1) for c in range(NUM_CLS) if r_c[c] > 1.5})
for c in range(NUM_CLS):
    if cls_count[("train", CLASSES[c])]: cls_count[("train_after_rfs", CLASSES[c])] = cls_count[("train", CLASSES[c])]
added = 0
for lf, present in list(img_classes.items()):
    if not present: continue
    r = min(REPEAT_CAP, max(r_c[c] for c in present))
    if r < 1.5: continue
    stem = Path(lf).stem
    imgs = glob.glob(f"{DATA}/train/images/{stem}.*")
    if not imgs: continue
    ext = Path(imgs[0]).suffix
    for k in range(int(round(r)) - 1):
        dst = f"{DATA}/train/images/{stem}_rfs{k}{ext}"
        if not os.path.exists(dst):
            shutil.copy(imgs[0], dst)
            shutil.copy(f"{DATA}/train/labels/{lf}", f"{DATA}/train/labels/{stem}_rfs{k}.txt")
        for c, cnt in present.items(): cls_count[("train_after_rfs", CLASSES[c])] += cnt
        added += 1
print(f"Oversampling: +{added} bản sao ảnh chứa lớp hiếm (train {n_train} -> {n_train+added})")

names_yaml = "\n".join(f"  {i}: {nm}" for i, nm in enumerate(CLASSES))
with open(f"{DATA}/data.yaml", "w") as f:
    f.write(f"path: {DATA}\ntrain: train/images\nval: val/images\ntest: test/images\nnames:\n{names_yaml}\n")

import pandas as pd
df_dist = pd.DataFrame([{"split":k[0],"class":k[1],"boxes":v} for k,v in cls_count.items()]).pivot_table(index="class",columns="split",values="boxes",fill_value=0)
df_dist.to_csv(f"{WORK}/results/dataset_distribution.csv"); df_dist

In [ ]:
import cv2
from PIL import Image

def build_crops(split, size):
    X, y = [], []; per_class = Counter()
    files = sorted(os.listdir(f"{DATA}/{split}/labels"))
    random.Random(SEED).shuffle(files)
    for lf in files:
        stem = Path(lf).stem
        if "_rfs" in stem: continue
        for ext in [".png",".jpg",".jpeg"]:
            ip = f"{DATA}/{split}/images/{stem}{ext}"
            if os.path.exists(ip): break
        else: continue
        img = Image.open(ip).convert("RGB"); W,H = img.size
        for line in open(f"{DATA}/{split}/labels/{lf}"):
            if not line.strip(): continue
            c,xc,yc,bw,bh = map(float, line.split()); c=int(c)
            if per_class[c] >= MAX_CROPS_PER_CLASS: continue
            w,h = bw*W, bh*H
            if w<16 or h<16: continue
            pad=0.15
            box = (max(0,(xc-bw/2)*W-pad*w), max(0,(yc-bh/2)*H-pad*h),
                   min(W,(xc+bw/2)*W+pad*w), min(H,(yc+bh/2)*H+pad*h))
            X.append(np.asarray(img.crop(box).resize((size,size)), dtype=np.uint8)); y.append(c)
            per_class[c]+=1
    return np.array(X), np.array(y)

crops = {sp: build_crops(sp, CROP_SIZE) for sp in ["train","val","test"]}
for sp in crops:
    print(sp, crops[sp][0].shape, {CLASSES[i]: int(v) for i,v in enumerate(np.bincount(crops[sp][1], minlength=NUM_CLS))})
(Xtr,ytr),(Xva,yva),(Xte,yte) = crops["train"], crops["val"], crops["test"]

from sklearn.utils.class_weight import compute_class_weight
present_cls = np.unique(ytr)
w = compute_class_weight("balanced", classes=present_cls, y=ytr)
CLASS_WEIGHT = {int(c): float(v) for c, v in zip(present_cls, w)}
CW_TENSOR = np.ones(NUM_CLS, dtype="float32")
for c, v in CLASS_WEIGHT.items(): CW_TENSOR[c] = v
print("Class weights:", {CLASSES[c]: round(v,2) for c,v in CLASS_WEIGHT.items()})

from sklearn.metrics import accuracy_score, f1_score, classification_report
def multiclass_metrics(y_true, pred):
    acc = accuracy_score(y_true, pred)
    macro = f1_score(y_true, pred, average="macro")
    per_cls = f1_score(y_true, pred, average=None, labels=list(range(NUM_CLS)), zero_division=0)
    support = np.bincount(y_true, minlength=NUM_CLS)
    valid = [c for c in range(NUM_CLS) if support[c] >= 10]   # lớp <10 mẫu (vd no_boots) loại khỏi worst - báo cáo riêng
    worst = float(np.min([per_cls[c] for c in valid])) if valid else 0.0
    return acc, macro, worst

## 1. YOLO26s – HPO (sàng lọc trên VAL, 10 epochs @512) → train cuối (40 epochs @640)

In [ ]:
from ultralytics import YOLO
IMG_FINAL, IMG_SCREEN = 640, 512
EP_SCREEN = 2 if QUICK_TEST else 5
EP_FINAL  = 3 if QUICK_TEST else 30

def load_det(name):
    try: return YOLO(name)
    except Exception as e:
        print(f"Không tải được {name} ({e}) -> fallback yolov8s.pt"); return YOLO("yolov8s.pt")

YOLO_GRID = [
    {"lr0": 0.01,  "optimizer": "auto"},   # MuSGD mặc định của YOLO26
    {"lr0": 0.01,  "optimizer": "SGD"},
    {"lr0": 0.005, "optimizer": "SGD"},
    {"lr0": 0.001, "optimizer": "AdamW"},
]
rows = []; best_cfg_yolo = YOLO_GRID[0]
if RUN_HPO:
    for i, cfg in enumerate(YOLO_GRID):
        key = f"yolo_hpo{i}"
        if done(key): rows.append(STATE[key]); print("skip:", STATE[key]); continue
        m = load_det("yolo26s.pt"); t0=time.time()
        m.train(data=f"{DATA}/data.yaml", epochs=EP_SCREEN, imgsz=IMG_SCREEN, batch=16,
                seed=SEED, project=f"{WORK}/runs", name=f"yolo_hpo{i}", plots=False, verbose=False,
                exist_ok=True, **cfg)
        v = m.val(data=f"{DATA}/data.yaml", split="val", verbose=False)
        row = {**cfg, "mAP50_val": round(v.box.map50,4), "min": round((time.time()-t0)/60,1)}
        rows.append(row); mark(key, row); print(row); del m; free()
    hpo_yolo = pd.DataFrame(rows).sort_values("mAP50_val", ascending=False)
    HPO_TABLES["YOLO26s"] = hpo_yolo
    best_cfg_yolo = {k: hpo_yolo.iloc[0][k] for k in ["lr0","optimizer"]}
    display(hpo_yolo)
print("Cấu hình chọn:", best_cfg_yolo)

In [ ]:
res_det = {}
if done("yolo_final"):
    res_det["YOLO26s (CNN)"] = STATE["yolo_final"]; print("skip:", STATE["yolo_final"])
else:
    last_ckpt = f"{WORK}/runs/yolo_final/weights/last.pt"; t0=time.time()
    if os.path.exists(last_ckpt):
        print("RESUME từ", last_ckpt); yolo = YOLO(last_ckpt); yolo.train(resume=True)
    else:
        yolo = load_det("yolo26s.pt")
        yolo.train(data=f"{DATA}/data.yaml", epochs=EP_FINAL, imgsz=IMG_FINAL, batch=16,
                   seed=SEED, project=f"{WORK}/runs", name="yolo_final", patience=10, plots=True,
                   exist_ok=True, lr0=float(best_cfg_yolo["lr0"]), optimizer=str(best_cfg_yolo["optimizer"]))
    mv = yolo.val(data=f"{DATA}/data.yaml", split="val")
    mt = yolo.val(data=f"{DATA}/data.yaml", split="test")
    res_det["YOLO26s (CNN)"] = {"cfg": str(best_cfg_yolo),
        "mAP50_val": round(mv.box.map50,4), "mAP50_test": round(mt.box.map50,4),
        "mAP50_95_test": round(mt.box.map,4), "precision_test": round(mt.box.mp,4),
        "recall_test": round(mt.box.mr,4), "infer_ms": round(mt.speed["inference"],2),
        "train_min": round((time.time()-t0)/60,1)}
    print("mAP50 từng lớp:", dict(zip([CLASSES[i] for i in mt.box.ap_class_index], np.round(mt.box.ap50,4))))
    mark("yolo_final", res_det["YOLO26s (CNN)"]); del yolo; free()
res_det["YOLO26s (CNN)"]

## 2. Faster R-CNN ResNet50-FPN (two-stage, torchvision) – baseline tham chiếu
Một cấu hình chuẩn (lr=0.005, SGD momentum, 12 epochs) — đại diện họ two-stage theo Session 03; đánh giá bằng torchmetrics mAP.

In [ ]:
FRCNN_EPOCHS = 1 if QUICK_TEST else 8
if done("frcnn_final"):
    res_det["Faster R-CNN (CNN)"] = STATE["frcnn_final"]; print("skip:", STATE["frcnn_final"])
else:
    import torchvision
    import torchvision.transforms.functional as TF
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    from torchmetrics.detection.mean_ap import MeanAveragePrecision
    from torch.utils.data import Dataset, DataLoader

    class DetDS(Dataset):
        def __init__(self, split):
            self.idir=f"{DATA}/{split}/images"; self.ldir=f"{DATA}/{split}/labels"
            self.files=[f for f in sorted(os.listdir(self.idir))]
        def __len__(self): return len(self.files)
        def __getitem__(self, i):
            f=self.files[i]; img=Image.open(f"{self.idir}/{f}").convert("RGB"); W,H=img.size
            boxes,labels=[],[]
            lp=f"{self.ldir}/{Path(f).stem}.txt"
            if os.path.exists(lp):
                for line in open(lp):
                    if not line.strip(): continue
                    c,xc,yc,bw,bh=map(float,line.split())
                    x1,y1,x2,y2=(xc-bw/2)*W,(yc-bh/2)*H,(xc+bw/2)*W,(yc+bh/2)*H
                    if x2>x1+1 and y2>y1+1: boxes.append([x1,y1,x2,y2]); labels.append(int(c)+1)  # 0 = background
            t={"boxes":torch.tensor(boxes,dtype=torch.float32).reshape(-1,4),
               "labels":torch.tensor(labels,dtype=torch.int64)}
            return TF.to_tensor(img), t
    def collate(b): return tuple(zip(*b))

    device="cuda"
    frcnn = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_f = frcnn.roi_heads.box_predictor.cls_score.in_features
    frcnn.roi_heads.box_predictor = FastRCNNPredictor(in_f, NUM_CLS+1)
    frcnn.to(device)
    opt = torch.optim.SGD([p for p in frcnn.parameters() if p.requires_grad], lr=0.005, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=8, gamma=0.1)
    dl_tr = DataLoader(DetDS("train"), batch_size=4, shuffle=True, collate_fn=collate, num_workers=2)
    t0=time.time()
    for ep in range(FRCNN_EPOCHS):
        frcnn.train(); tot=0
        for imgs, targets in dl_tr:
            imgs=[i.to(device) for i in imgs]
            targets=[{k:v.to(device) for k,v in t.items()} for t in targets]
            loss = sum(frcnn(imgs, targets).values())
            opt.zero_grad(); loss.backward(); opt.step(); tot+=float(loss)
        sched.step(); print(f"epoch {ep+1}/{FRCNN_EPOCHS} loss={tot/len(dl_tr):.3f}")

    @torch.no_grad()
    def frcnn_eval(split):
        frcnn.eval(); metric = MeanAveragePrecision(iou_type="bbox")
        dl = DataLoader(DetDS(split), batch_size=4, shuffle=False, collate_fn=collate, num_workers=2)
        n_img=0; t_inf=0.
        for imgs, targets in dl:
            imgs=[i.to(device) for i in imgs]; s=time.time()
            preds = frcnn(imgs); t_inf += time.time()-s; n_img += len(imgs)
            metric.update([{k:v.cpu() for k,v in p.items()} for p in preds], targets)
        r = metric.compute()
        return float(r["map_50"]), float(r["map"]), t_inf/max(1,n_img)*1000
    map50_v,_,_ = frcnn_eval("val")
    map50_t, map5095_t, ms = frcnn_eval("test")
    res_det["Faster R-CNN (CNN)"] = {"cfg": "lr=0.005,SGD,8ep",
        "mAP50_val": round(map50_v,4), "mAP50_test": round(map50_t,4),
        "mAP50_95_test": round(map5095_t,4), "infer_ms": round(ms,2),
        "train_min": round((time.time()-t0)/60,1)}
    mark("frcnn_final", res_det["Faster R-CNN (CNN)"])
    print(res_det["Faster R-CNN (CNN)"]); del frcnn; free()

## 3. VGG16 & ResNet50 (Keras) – classification crop 10 lớp
HPO: backbone × lr × dropout (8 cấu hình sàng lọc 6 epochs) → train cuối best của TỪNG backbone (+ unfreeze block cuối, class weights).

In [ ]:
import tensorflow as tf
for g in tf.config.list_physical_devices('GPU'): tf.config.experimental.set_memory_growth(g, True)
from tensorflow import keras
from tensorflow.keras import layers

EP_SCREEN_CLS = 2 if QUICK_TEST else 6
EP_FINAL_CLS  = 3 if QUICK_TEST else 12

PRE = {"VGG16": keras.applications.vgg16.preprocess_input,
       "ResNet50": keras.applications.resnet50.preprocess_input}
def make_cnn(backbone, lr, dropout):
    if backbone == "VGG16":
        base = keras.applications.VGG16(weights="imagenet", include_top=False, input_shape=(CROP_SIZE,CROP_SIZE,3))
    else:
        base = keras.applications.ResNet50(weights="imagenet", include_top=False, input_shape=(CROP_SIZE,CROP_SIZE,3))
    base.trainable = False
    m = keras.Sequential([base, layers.GlobalAveragePooling2D(),
                          layers.Dense(256, activation="relu"), layers.Dropout(dropout),
                          layers.Dense(NUM_CLS, activation="softmax")])
    m.compile(optimizer=keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m, base
def cbs(): return [keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
                   keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.3)]

CNN_GRID = [{"backbone":b,"lr":lr,"dropout":d} for b in ["VGG16","ResNet50"] for lr in [1e-3,1e-4] for d in [0.3,0.5]]
rows=[]
if RUN_HPO:
    for i, cfg in enumerate(CNN_GRID):
        key = f"cnn_hpo{i}"
        if done(key): rows.append(STATE[key]); print("skip:", STATE[key]); continue
        pp = PRE[cfg["backbone"]]
        m,_ = make_cnn(**cfg); t0=time.time()
        m.fit(pp(Xtr.astype("float32")), ytr, validation_data=(pp(Xva.astype("float32")), yva),
              epochs=EP_SCREEN_CLS, batch_size=64, class_weight=CLASS_WEIGHT, verbose=0)
        pred = m.predict(pp(Xva.astype("float32")), verbose=0).argmax(1)
        acc, macro, worst = multiclass_metrics(yva, pred)
        row = {**cfg, "macro_f1_val": round(macro,4), "min": round((time.time()-t0)/60,1)}
        rows.append(row); mark(key, row); print(row); keras.backend.clear_session()
    hpo_cnn = pd.DataFrame(rows).sort_values("macro_f1_val", ascending=False)
    HPO_TABLES["CNN_cls"] = hpo_cnn; display(hpo_cnn)

res_cls = {}
for backbone in ["VGG16","ResNet50"]:
    key = f"{backbone}_final"
    if done(key): res_cls[f"{backbone} (CNN)"] = STATE[key]; print("skip:", STATE[key]); continue
    if RUN_HPO:
        sub = hpo_cnn[hpo_cnn["backbone"]==backbone].iloc[0]
        cfg = {"backbone":backbone, "lr":float(sub["lr"]), "dropout":float(sub["dropout"])}
    else:
        cfg = {"backbone":backbone, "lr":1e-3, "dropout":0.5}
    pp = PRE[backbone]
    Xtr_p, Xva_p, Xte_p = pp(Xtr.astype("float32")), pp(Xva.astype("float32")), pp(Xte.astype("float32"))
    m, base = make_cnn(**cfg); t0=time.time()
    m.fit(Xtr_p, ytr, validation_data=(Xva_p,yva), epochs=EP_FINAL_CLS, batch_size=64,
          class_weight=CLASS_WEIGHT, callbacks=cbs(), verbose=2)
    tail = "conv5" if backbone=="ResNet50" else "block5"
    for l in base.layers: l.trainable = l.name.startswith(tail)
    m.compile(optimizer=keras.optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    m.fit(Xtr_p, ytr, validation_data=(Xva_p,yva), epochs=max(3,EP_FINAL_CLS//3), batch_size=64,
          class_weight=CLASS_WEIGHT, callbacks=cbs(), verbose=2)
    pred = m.predict(Xte_p, verbose=0).argmax(1)
    acc, macro, worst = multiclass_metrics(yte, pred)
    res_cls[f"{backbone} (CNN)"] = {"cfg": str(cfg)+f"+unfreeze {tail}",
        "accuracy":round(acc,4), "macro_f1":round(macro,4), "worst_class_f1":round(worst,4),
        "train_min":round((time.time()-t0)/60,1)}
    print(res_cls[f"{backbone} (CNN)"])
    print(classification_report(yte, pred, labels=list(range(NUM_CLS)), target_names=CLASSES, zero_division=0))
    m.save(f"{WORK}/results/{backbone.lower()}_best.keras")
    mark(key, res_cls[f"{backbone} (CNN)"]); keras.backend.clear_session()

## 4. Xuất kết quả chuẩn hóa

In [ ]:
# ====== XUẤT summary.csv chuẩn hóa để notebook 04 tổng hợp ======
rows = []
for name, d in res_det.items():
    rows.append({"branch": BRANCH, "task": "detection", "model": name, **d})
for name, d in res_cls.items():
    rows.append({"branch": BRANCH, "task": "classification", "model": name,
                 **{k: v for k, v in d.items() if k != "_name"}})
df_sum = pd.DataFrame(rows)
df_sum.to_csv(f"{WORK}/results/summary.csv", index=False)
for name, tbl in HPO_TABLES.items():
    tbl.to_csv(f"{WORK}/results/hpo_{name.replace('/','').replace(' ','_')}.csv", index=False)
print("Đã lưu results/summary.csv + các bảng HPO. Sau khi COMMIT xong, dùng output này làm Input cho notebook 04.")
display(df_sum)

## Checkpoint & Resume (nếu notebook này bị cắt trước khi xong)
Notebook lưu tiến độ vào `state.json` + `runs/` sau mỗi cấu hình/model. Nếu phiên bị cắt: (1) chạy bằng **Save & Run All (Commit)** nên output vẫn được giữ trong version đó; (2) mở notebook → **+ Add Input → Your Work** → chọn output version vừa dừng; (3) đặt `RESUME_FROM = "/kaggle/input/<slug>"` ở cell Config; (4) Run All lại — stage đã xong tự bỏ qua, detection train tiếp từ `last.pt`.